In [19]:
from jaad_data import JAAD

jaad_api = JAAD(data_path = '.')

In [20]:
# Extract & Save Images
# JAAD has in-built method extract_and_save_images, but it extracts images in png format, which takes up too much disk space.
import os
import cv2

def extract_all_images(jaad_obj, image_ext=".jpg", overwrite=False):
    clip_files = sorted(
        f for f in os.listdir(jaad_obj._clips_path)
        if f.lower().endswith(".mp4")
    )

    os.makedirs(jaad_obj._images_path, exist_ok=True)
    print(f"Found {len(clip_files)} clips in {jaad_obj._clips_path}")

    for i, clip_file in enumerate(clip_files, start=1):
        vid = os.path.splitext(clip_file)[0]
        clip_path = os.path.join(jaad_obj._clips_path, clip_file)
        save_dir = os.path.join(jaad_obj._images_path, vid)
        os.makedirs(save_dir, exist_ok=True)

        cap = cv2.VideoCapture(clip_path)
        if not cap.isOpened():
            print(f"[{i}/{len(clip_files)}] [SKIP] cannot open: {clip_path}")
            continue

        frame_idx = 0
        saved = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            out_path = os.path.join(save_dir, f"{frame_idx:05d}{image_ext}")
            if overwrite or not os.path.exists(out_path):
                cv2.imwrite(out_path, frame)
                saved += 1
            frame_idx += 1

        cap.release()
        print(f"[{i}/{len(clip_files)}] {vid}: saved {saved} / {frame_idx} frames")

# Run extraction for all videos if files do not already exist, else do not run
if not os.path.exists(jaad_api._images_path) or not os.listdir(jaad_api._images_path):
    extract_all_images(jaad_api, image_ext=".jpg", overwrite=False)
else:
    print(f"Images already exist in {jaad_api._images_path}, skipping extraction.")

Images already exist in .\images, skipping extraction.


In [21]:
# Generate Database
db = jaad_api.generate_database()

---------------------------------------------------------
Generating database for jaad
jaad database loaded from c:\Users\ASUS\Documents\Year 3 Semester 2\JAAD\data_cache\jaad_database.pkl


In [22]:
# Extract combined features (JAAD annotations + pose data from pedestrian_poses.pkl)
# Notes on meaning of Behavior Annotations:
    # occlusion: 0(not occluded), 1(partially occluded), 2(fully occluded)
    # cross: 0(not crossing), 1(crossing)
    # reaction: 0(no reaction), 1(reaction)
    # hand_gesture: 0(no hand gesture), 1(hand gesture)
    # look: 0(not looking), 1(looking)
    # action: 0(Standing), 1(Walking)
    # nod: 0(no nod), 1(nod)
# Notes on meaning of Pedestrian Attributes:
    # old_id: Original Annotation ID String
    # age: 0(child), 1 (young), 2 (adult), 3 (senior)
    # crossing: 0(not crossing), 1(crossing), -1(irrelevant)
    # crossing_point: Frame Index of Crossing Point (if crossing), -1 otherwise
    # decision_point: Frame Index of Decision Point (if crossing), -1 otherwise
    # designated: 0(not designated crossing point), 1(designated crossing point)
    # gender: 0(n/a), 1(female), 2(male)
    # group_size: Number of People in Group
    # intersection: 0(not at intersection), 1(at intersection)
    # motion_direction: 0(n/a), 1(Lateral / Across), 2(Longitudinal / Along)
    # num_lanes: Number of Road Lanes
    # signalized: 0(n/a), 1(non-signalized intersection), 2(signalized intersection)
    # traffic_direction: 0(One-Way), 1(Two-Way)
# Notes on pose data in pedestrian_poses.pkl:
    # pose_keypoints_xy: List of (x, y) coordinates for each keypoint.
    # pose_keypoints_conf: List of confidence scores for each keypoint.

import os
import pickle
import numpy as np
import pandas as pd

POSE_PKL_PATH = "./pedestrian_poses.pkl"
pedestrian_ids = jaad_api._get_pedestrian_ids()

# ----- Base JAAD tabular features -----
features = []
for vid, video in db.items():
    for pid, pedestrian in video["ped_annotations"].items():
        if 'b' not in pid:  # Skip pedestrians without behavior annotations
            continue

        frames = pedestrian.get("frames", [])
        bboxes = pedestrian.get("bbox", [])
        occlusions = pedestrian.get("occlusion", [])
        behavior = pedestrian.get("behavior", {})
        attributes = pedestrian.get("attributes", {})

        for i, frame in enumerate(frames):
            row = {
                "video_id": vid,
                "pedestrian_id": pid,
                "frame_id": int(frame),
                "bbox_x1": bboxes[i][0],
                "bbox_y1": bboxes[i][1],
                "bbox_x2": bboxes[i][2],
                "bbox_y2": bboxes[i][3],

                # Behaviour Annotations
                "occlusion": occlusions[i],
                "cross": behavior.get("cross", [0] * len(frames))[i],
                "reaction": behavior.get("reaction", [0] * len(frames))[i],
                "hand_gesture": behavior.get("hand_gesture", [0] * len(frames))[i],
                "look": behavior.get("look", [0] * len(frames))[i],
                "action": behavior.get("action", [0] * len(frames))[i],
                "nod": behavior.get("nod", [0] * len(frames))[i],

                # Pedestrian Attributes
                "old_id": attributes.get("old_id", ""),
                "age": attributes.get("age", 0),
                "crossing": attributes.get("crossing", 0),
                "crossing_point": attributes.get("crossing_point", 0),
                "decision_point": attributes.get("decision_point", 0),
                "designated": attributes.get("designated", 0),
                "gender": attributes.get("gender", 0),
                "group_size": attributes.get("group_size", 1),
                "intersection": attributes.get("intersection", 0),
                "motion_direction": attributes.get("motion_direction", 0),
                "num_lanes": attributes.get("num_lanes", 0),
                "signalized": attributes.get("signalized", 0),
                "traffic_direction": attributes.get("traffic_direction", 0),
            }
            features.append(row)

jaad_df = pd.DataFrame(features)

# ----- Raw pose records from pedestrian_poses.pkl -----
def load_raw_pose_frame_records(pkl_path=POSE_PKL_PATH):
    cols = [
        "video_id", "pedestrian_id", "frame_id",
        "pose_keypoints_xy", "pose_keypoints_conf"
    ]
    if not os.path.exists(pkl_path):
        print(f"[WARN] Pose file not found: {pkl_path}")
        return pd.DataFrame(columns=cols)

    with open(pkl_path, "rb") as f:
        poses_by_track = pickle.load(f)

    rows = []
    for records in poses_by_track.values():
        for rec in records:
            rows.append({
                "video_id": rec.get("video_id"),
                "pedestrian_id": rec.get("pedestrian_id"),
                "frame_id": int(rec.get("frame_id", -1)),
                "pose_keypoints_xy": rec.get("keypoints_xy"),
                "pose_keypoints_conf": rec.get("keypoints_conf"),
            })

    pose_df = pd.DataFrame(rows, columns=cols)
    # Keep one pose record per key; if duplicates exist, keep first raw record.
    pose_df = pose_df.drop_duplicates(subset=["video_id", "pedestrian_id", "frame_id"], keep="first")
    return pose_df

pose_raw_df = load_raw_pose_frame_records()

# Combined dataframe used downstream
jaad_df["frame_id"] = jaad_df["frame_id"].astype(int)
merged_features_df = jaad_df.merge(
    pose_raw_df,
    on=["video_id", "pedestrian_id", "frame_id"],
    how="left"
 )

# Convenience flags / aliases
merged_features_df["has_pose"] = merged_features_df["pose_keypoints_xy"].notna().astype(int)
df = merged_features_df
pose_features_df = pose_raw_df

print(f"Combined features shape: {df.shape}")
print(f"Rows with pose: {int(df['has_pose'].sum())}")
display(df.head())

# Count number of unique pedestrians
unique_pedestrians = df["pedestrian_id"].nunique()
assert unique_pedestrians == len([pid for pid in pedestrian_ids if 'b' in pid]), "Mismatch between unique pedestrians in features and pedestrian IDs."

# Check class imbalance in crossing behavior attribute
crossing_counts = df["crossing"].value_counts()
display(crossing_counts)

---------------------------------------------------------
Generating database for jaad
jaad database loaded from c:\Users\ASUS\Documents\Year 3 Semester 2\JAAD\data_cache\jaad_database.pkl
Combined features shape: (132700, 30)
Rows with pose: 103287


---------------------------------------------------------
Generating database for jaad
jaad database loaded from c:\Users\ASUS\Documents\Year 3 Semester 2\JAAD\data_cache\jaad_database.pkl
Combined features shape: (132700, 30)
Rows with pose: 103287


,video_id,pedestrian_id,frame_id,bbox_x1,bbox_y1,bbox_x2,bbox_y2,occlusion,cross,reaction,...,gender,group_size,intersection,motion_direction,num_lanes,signalized,traffic_direction,pose_keypoints_xy,pose_keypoints_conf,has_pose
0,video_0001,0_1_3b,0,465.0,730.0,533.0,848.0,0,0,0,...,1,1,0,2,2,0,1,"[[527.3539428710938, 745.8139038085938], [524....","[0.3591992259025574, 0.021447576582431793, 0.5...",1
1,video_0001,0_1_3b,1,463.0,730.0,532.0,848.0,0,0,0,...,1,1,0,2,2,0,1,"[[522.7257080078125, 742.2144775390625], [522....","[0.7889110445976257, 0.21608410775661469, 0.82...",1
2,video_0001,0_1_3b,2,461.0,730.0,531.0,849.0,0,0,0,...,1,1,0,2,2,0,1,"[[523.2115478515625, 744.8243408203125], [521....","[0.4840393364429474, 0.02693798393011093, 0.80...",1
3,video_0001,0_1_3b,3,459.0,730.0,530.0,849.0,0,0,0,...,1,1,0,2,2,0,1,"[[522.5260620117188, 744.585205078125], [521.1...","[0.37200379371643066, 0.02177983894944191, 0.6...",1
4,video_0001,0_1_3b,4,458.0,731.0,530.0,851.0,0,0,0,...,1,1,0,2,2,0,1,"[[519.7975463867188, 745.635986328125], [517.7...","[0.21326614916324615, 0.01605609431862831, 0.4...",1


crossing
 1    105026
-1     16216
 0     11458
Name: count, dtype: int64

In [23]:
# Train XGBoost classifier to predict crossing behavior (frame-level binary classification)

import os
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_curve, classification_report, roc_auc_score
from xgboost import XGBClassifier

# Safety-focused threshold shared by evaluation and video rendering
DECISION_THRESHOLD = 0.15

# Keep names consistent for training/inference/video rendering
cat_cols = [
    "occlusion", "reaction", "hand_gesture", "look", "action", "nod",
    "age", "designated", "gender", "intersection",
    "motion_direction", "signalized", "traffic_direction"
]
num_cols = [
    "bbox_center_x", "bbox_center_y", "bbox_width", "bbox_height", "bbox_area",
    "velocity_x", "velocity_y", "speed", "acceleration_x", "acceleration_y",
    "group_size", "num_lanes",
    "pose_visible_kpts", "pose_mean_conf", "pose_x_span", "pose_y_span"
]


def engineer_jaad_features(frame_df):
    """Compute derived geometry and motion features from JAAD bbox tracks."""
    out = frame_df.copy()

    # JAAD frames are 1920 x 1080
    W, H = 1920.0, 1080.0

    # Normalized bbox geometry
    out["bbox_center_x"] = ((out["bbox_x1"] + out["bbox_x2"]) / 2.0) / W
    out["bbox_center_y"] = ((out["bbox_y1"] + out["bbox_y2"]) / 2.0) / H
    out["bbox_width"] = (out["bbox_x2"] - out["bbox_x1"]) / W
    out["bbox_height"] = (out["bbox_y2"] - out["bbox_y1"]) / H
    out["bbox_area"] = out["bbox_width"] * out["bbox_height"]

    # Pedestrian velocity and acceleration (frame-to-frame differences)
    group_keys = ["video_id", "pedestrian_id"]
    out["velocity_x"] = out.groupby(group_keys)["bbox_center_x"].diff().fillna(0.0)
    out["velocity_y"] = out.groupby(group_keys)["bbox_center_y"].diff().fillna(0.0)
    out["speed"] = np.sqrt(out["velocity_x"] ** 2 + out["velocity_y"] ** 2)
    out["acceleration_x"] = out.groupby(group_keys)["velocity_x"].diff().fillna(0.0)
    out["acceleration_y"] = out.groupby(group_keys)["velocity_y"].diff().fillna(0.0)

    return out


def engineer_pose_features(frame_df, conf_threshold=0.30):
    """Compute compact pose features from raw keypoints and confidences."""
    out = frame_df.copy()
    out["pose_visible_kpts"] = 0.0
    out["pose_mean_conf"] = 0.0
    out["pose_x_span"] = 0.0
    out["pose_y_span"] = 0.0

    if "pose_keypoints_xy" not in out.columns:
        return out

    for idx in out.index:
        kxy_raw = out.at[idx, "pose_keypoints_xy"]
        kcf_raw = out.at[idx, "pose_keypoints_conf"] if "pose_keypoints_conf" in out.columns else None

        if kxy_raw is None or (isinstance(kxy_raw, float) and np.isnan(kxy_raw)):
            continue

        kxy = np.asarray(kxy_raw, dtype=float)
        if kxy.ndim != 2 or kxy.shape[1] != 2 or kxy.shape[0] == 0:
            continue

        kcf = np.asarray(kcf_raw, dtype=float) if kcf_raw is not None else np.array([])
        if kcf.size == 0:
            kcf = np.ones(kxy.shape[0], dtype=float)

        if kcf.shape[0] != kxy.shape[0]:
            n = min(kcf.shape[0], kxy.shape[0])
            if n == 0:
                continue
            kcf = kcf[:n]
            kxy = kxy[:n]

        out.at[idx, "pose_visible_kpts"] = float((kcf > conf_threshold).sum())
        out.at[idx, "pose_mean_conf"] = float(kcf.mean()) if kcf.size else 0.0
        out.at[idx, "pose_x_span"] = float(kxy[:, 0].max() - kxy[:, 0].min())
        out.at[idx, "pose_y_span"] = float(kxy[:, 1].max() - kxy[:, 1].min())

    return out


def read_ids(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


# ----- DATA PREP -----
source_df = df.copy() if "df" in globals() else pd.DataFrame(features).copy()
df = source_df.sort_values(["video_id", "pedestrian_id", "frame_id"]).reset_index(drop=True)

# ----- FEATURE ENGINEERING -----
df = engineer_jaad_features(df)
df = engineer_pose_features(df, conf_threshold=0.30)

# Frame-level binary target only
df = df[df["cross"].isin([0, 1])].copy()

# ----- DATA SPLIT -----
split_root = os.path.join("split_ids", "default")
train_videos = read_ids(os.path.join(split_root, "train.txt"))
val_videos = read_ids(os.path.join(split_root, "val.txt"))
test_videos = read_ids(os.path.join(split_root, "test.txt"))
assert set(train_videos).isdisjoint(set(val_videos)), "Train and validation video IDs overlap."
assert set(train_videos).isdisjoint(set(test_videos)), "Train and test video IDs overlap."
assert set(val_videos).isdisjoint(set(test_videos)), "Validation and test video IDs overlap."

train_df = df[df["video_id"].isin(train_videos)].copy()
val_df = df[df["video_id"].isin(val_videos)].copy()
test_df = df[df["video_id"].isin(test_videos)].copy()
assert len(set(train_df.index).intersection(val_df.index)) == 0, "Train and validation frame indices overlap."
assert len(set(train_df.index).intersection(test_df.index)) == 0, "Train and test frame indices overlap."
assert len(set(val_df.index).intersection(test_df.index)) == 0, "Validation and test frame indices overlap."

print("Frame counts by split:")
print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})

X_train = train_df[cat_cols + num_cols]
y_train = train_df["cross"]
X_val = val_df[cat_cols + num_cols]
y_val = val_df["cross"]
X_test = test_df[cat_cols + num_cols]
y_test = test_df["cross"]

# Handle imbalance using training split only
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = float(neg) / float(pos) if pos > 0 else 1.0
print(f"scale_pos_weight: {scale_pos_weight:.3f}")

# ----- MODEL TRAINING -----
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols)
])

clf = Pipeline([
    ("pre", pre),
    ("model", XGBClassifier(
        n_estimators=2500,
        max_depth=8,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
    ))
])

clf.fit(X_train, y_train)

# ----- EVALUATION -----
# Validation metrics
val_proba = clf.predict_proba(X_val)[:, 1]

val_pred = (val_proba >= DECISION_THRESHOLD).astype(int)
print(f"\nValidation threshold: {DECISION_THRESHOLD}")
print("Validation AUROC:", roc_auc_score(y_val, val_proba))
print(classification_report(y_val, val_pred))

# Test metrics
test_proba = clf.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= DECISION_THRESHOLD).astype(int)
print("Test AUROC:", roc_auc_score(y_test, test_proba))
print(classification_report(y_test, test_pred))

Frame counts by split:
{'train': 61805, 'val': 9583, 'test': 52966}
scale_pos_weight: 0.722

Validation threshold: 0.15
Validation AUROC: 0.9604345044270552
              precision    recall  f1-score   support

           0       0.97      0.79      0.87      4325
           1       0.85      0.98      0.91      5258

    accuracy                           0.89      9583
   macro avg       0.91      0.89      0.89      9583
weighted avg       0.90      0.89      0.89      9583

Test AUROC: 0.9083269998345477
              precision    recall  f1-score   support

           0       0.93      0.67      0.78     23243
           1       0.79      0.96      0.87     29723

    accuracy                           0.83     52966
   macro avg       0.86      0.81      0.82     52966
weighted avg       0.85      0.83      0.83     52966



In [24]:
# Visualization of predictions on test video frames for XGBoost classifier
import pandas as pd
import cv2
import os
import numpy as np

# Temporal post-processing parameters to reduce frame-to-frame prediction flicker
# Use online mode for real-time inference (causal: uses only current/past frames).
TEMP_SMOOTHING_MODE = "online"  # "online" (causal) or "offline" (centered, uses future frames)
TEMP_SMOOTH_WINDOW = 5
TEMP_ENTER_THRESHOLD = 0.15
TEMP_EXIT_THRESHOLD = 0.10
TEMP_MIN_RUN_LENGTH = 2

# Split IDs are needed by both rendering and QA (QA can run without rendering).
test_ids_path = os.path.join("split_ids", "default", "test.txt")
test_video_ids = read_ids(test_ids_path)

# Toggle expensive rendering work; keep False when you only want QA/stat analysis.
RENDER_TEST_VIDEOS = False


def _remove_short_runs(labels, min_run_length=4):
    """Remove short on/off bursts by merging them into neighboring runs."""
    arr = np.asarray(labels, dtype=int).copy()
    n = arr.size
    if n == 0 or min_run_length <= 1:
        return arr

    start = 0
    while start < n:
        end = start + 1
        while end < n and arr[end] == arr[start]:
            end += 1

        run_len = end - start
        if run_len < min_run_length:
            left_val = arr[start - 1] if start > 0 else None
            right_val = arr[end] if end < n else None

            if left_val is None and right_val is None:
                pass
            elif left_val is None:
                arr[start:end] = right_val
            elif right_val is None:
                arr[start:end] = left_val
            elif left_val == right_val:
                arr[start:end] = left_val
            else:
                arr[start:end] = left_val

        start = end

    return arr


def _temporal_filter_track(
    track_df,
    score_col="crossing_score",
    smooth_window=7,
    enter_th=0.30,
    exit_th=0.20,
    min_run_length=4,
    threshold_fallback=0.25,
    smoothing_mode="online",
):
    """Apply smoothing + hysteresis + run-length cleanup to one track."""
    out = track_df.sort_values("frame_id").copy()

    mode = str(smoothing_mode).lower().strip()
    if mode not in {"online", "offline"}:
        raise ValueError("smoothing_mode must be 'online' or 'offline'.")

    # online: causal smoothing for real-time inference (no future frames)
    # offline: centered smoothing for post-processed analysis/videos
    is_centered = (mode == "offline")
    out["crossing_score_smooth"] = (
        out[score_col].rolling(window=smooth_window, min_periods=1, center=is_centered).mean()
    )

    state = 0
    preds = []
    for score in out["crossing_score_smooth"].to_numpy(dtype=float):
        if state == 0 and score >= enter_th:
            state = 1
        elif state == 1 and score < exit_th:
            state = 0
        preds.append(state)

    preds = _remove_short_runs(preds, min_run_length=min_run_length)
    out["predicted_cross_temporal"] = preds.astype(int)

    # Keep a fallback binary output for compatibility/debugging
    out["predicted_cross"] = (out[score_col] >= threshold_fallback).astype(int)
    return out


def predict_from_df(
    df,
    clf,
    cat_cols,
    num_cols,
    threshold,
    video_id=None,
    pedestrian_id=None,
    use_temporal_filter=True,
    smoothing_mode=TEMP_SMOOTHING_MODE,
    smooth_window=TEMP_SMOOTH_WINDOW,
    enter_threshold=TEMP_ENTER_THRESHOLD,
    exit_threshold=TEMP_EXIT_THRESHOLD,
    min_run_length=TEMP_MIN_RUN_LENGTH,
):
    """Return per-frame crossing probabilities from engineered features with optional temporal filtering."""
    data = df.copy()

    if video_id is not None:
        data = data[data["video_id"] == video_id].copy()
    if pedestrian_id is not None:
        data = data[data["pedestrian_id"] == pedestrian_id].copy()

    if data.empty:
        raise ValueError("No rows matched the requested video_id / pedestrian_id filter.")

    data = data.sort_values(["video_id", "pedestrian_id", "frame_id"]).reset_index(drop=True)

    required_cols = set(cat_cols + num_cols)
    missing_cols = sorted(required_cols.difference(data.columns))
    if missing_cols:
        raise ValueError(
            "Cell 6 expects the dataframe to already be engineered by Cell 5. "
            f"Missing columns: {missing_cols}"
        )

    X = data.reindex(columns=cat_cols + num_cols, fill_value=0)
    scores = clf.predict_proba(X)[:, 1]
    result = data[[
        "video_id", "pedestrian_id", "frame_id",
        "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2",
        "cross"
    ]].copy()
    result["crossing_score"] = scores
    result["predicted_cross"] = (scores >= threshold).astype(int)

    if use_temporal_filter:
        result = (
            result.groupby(["video_id", "pedestrian_id"], group_keys=False)
            .apply(
                _temporal_filter_track,
                score_col="crossing_score",
                smooth_window=smooth_window,
                enter_th=enter_threshold,
                exit_th=exit_threshold,
                min_run_length=min_run_length,
                threshold_fallback=threshold,
                smoothing_mode=smoothing_mode,
            )
            .reset_index(drop=True)
        )
    else:
        result["crossing_score_smooth"] = result["crossing_score"]
        result["predicted_cross_temporal"] = result["predicted_cross"]

    return result


def write_prediction_video_from_df(
    df,
    obj,
    clf,
    cat_cols,
    num_cols,
    threshold,
    video_id,
    output_path,
    fps=30,
    use_temporal_filter=True,
    smoothing_mode=TEMP_SMOOTHING_MODE,
    smooth_window=TEMP_SMOOTH_WINDOW,
    enter_threshold=TEMP_ENTER_THRESHOLD,
    exit_threshold=TEMP_EXIT_THRESHOLD,
    min_run_length=TEMP_MIN_RUN_LENGTH,
):
    """Write a video with bounding boxes, temporal prediction, and ground truth per frame."""
    predictions = predict_from_df(
        df,
        clf,
        cat_cols,
        num_cols,
        threshold=threshold,
        video_id=video_id,
        use_temporal_filter=use_temporal_filter,
        smoothing_mode=smoothing_mode,
        smooth_window=smooth_window,
        enter_threshold=enter_threshold,
        exit_threshold=exit_threshold,
        min_run_length=min_run_length,
    )

    images_root = getattr(obj, "_images_path", os.path.join(".", "images"))
    video_path = os.path.join(images_root, video_id)
    if not os.path.isdir(video_path):
        raise FileNotFoundError(f"Video image folder not found: {video_path}")

    frame_files = sorted([
        f for f in os.listdir(video_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])
    if not frame_files:
        raise RuntimeError(f"No frames found in {video_path}. Expected .jpg or .png files.")

    first_frame = cv2.imread(os.path.join(video_path, frame_files[0]))
    if first_frame is None:
        raise RuntimeError(f"Could not read first frame: {frame_files[0]}")

    height, width = first_frame.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    if not writer.isOpened():
        raise RuntimeError(f"Could not open video writer for: {output_path}")

    pred_by_frame = {}
    for _, row in predictions.iterrows():
        pred_by_frame.setdefault(int(row["frame_id"]), []).append(row)

    for frame_file in frame_files:
        frame_idx = int(os.path.splitext(frame_file)[0])
        img = cv2.imread(os.path.join(video_path, frame_file))
        if img is None:
            continue

        for row in pred_by_frame.get(frame_idx, []):
            bbox = [row["bbox_x1"], row["bbox_y1"], row["bbox_x2"], row["bbox_y2"]]
            score = float(row["crossing_score"])
            score_smooth = float(row["crossing_score_smooth"])
            pred_label = int(row["predicted_cross_temporal"])
            gt_label = int(row["cross"])
            color = (0, int(255 * (1 - score_smooth)), int(255 * score_smooth))
            cv2.rectangle(img, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), color, 3)
            label = (
                f"{row['pedestrian_id']} Pred:{pred_label} GT:{gt_label} "
                f"Raw:{score:.2f} Sm:{score_smooth:.2f}"
            )
            cv2.putText(
                img, label, (int(bbox[0]), max(20, int(bbox[1] - 10))),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2
            )

        writer.write(img)

    writer.release()
    return output_path


print("Temporal filter config:")
print({
    "smoothing_mode": TEMP_SMOOTHING_MODE,
    "smooth_window": TEMP_SMOOTH_WINDOW,
    "enter_threshold": TEMP_ENTER_THRESHOLD,
    "exit_threshold": TEMP_EXIT_THRESHOLD,
    "min_run_length": TEMP_MIN_RUN_LENGTH,
    "render_test_videos": RENDER_TEST_VIDEOS,
})

if RENDER_TEST_VIDEOS:
    # Build videos for TEST split only using temporal-stabilized predictions
    output_dir = os.path.join(".", "predictions_test_videos")
    os.makedirs(output_dir, exist_ok=True)

    written = []
    skipped = []
    for video_id in test_video_ids:
        output_video_path = os.path.join(output_dir, f"{video_id}_predictions.mp4")
        try:
            write_prediction_video_from_df(
                df,
                jaad_api,
                clf,
                cat_cols,
                num_cols,
                threshold=DECISION_THRESHOLD,
                video_id=video_id,
                output_path=output_video_path,
                use_temporal_filter=True,
                smoothing_mode=TEMP_SMOOTHING_MODE,
                smooth_window=TEMP_SMOOTH_WINDOW,
                enter_threshold=TEMP_ENTER_THRESHOLD,
                exit_threshold=TEMP_EXIT_THRESHOLD,
                min_run_length=TEMP_MIN_RUN_LENGTH,
            )
            written.append(output_video_path)
            print(f"[OK] {video_id} -> {output_video_path}")
        except Exception as e:
            skipped.append((video_id, str(e)))
            print(f"[SKIP] {video_id}: {e}")

    print(f"\nCreated {len(written)} test videos.")
    print(f"Skipped {len(skipped)} test videos.")
    if skipped:
        display(pd.DataFrame(skipped, columns=["video_id", "reason"]))
else:
    print("Skipping video rendering. Functions/config/test IDs are ready for QA.")

Temporal filter config:
{'smoothing_mode': 'online', 'smooth_window': 5, 'enter_threshold': 0.15, 'exit_threshold': 0.1, 'min_run_length': 2, 'render_test_videos': False}
Skipping video rendering. Functions/config/test IDs are ready for QA.


In [25]:
# QA / statistical safety analysis: how early predictions are made and whether that is enough to stop
import numpy as np
import pandas as pd

# -------------------- Config --------------------
ANALYSIS_FPS = 30.0
ANALYSIS_USE_TEMPORAL = True

# Vehicle / braking assumptions (editable)
REACTION_TIME_SEC = 1.0
DECEL_MPS2 = 6.5
SPEEDS_KMH = [30, 40, 50]

# Optional filter: require a minimum sustained crossing run for GT onset
MIN_GT_ONSET_RUN = 3


def first_onset_frame(frame_ids, labels, min_run=1):
    """Return first 0->1 onset frame. Optionally require at least min_run consecutive ones."""
    f = np.asarray(frame_ids, dtype=int)
    y = np.asarray(labels, dtype=int)
    n = len(y)
    if n == 0:
        return None

    for i in range(n):
        if y[i] != 1:
            continue
        if i > 0 and y[i - 1] == 1:
            continue
        j = i
        while j < n and y[j] == 1:
            j += 1
        if (j - i) >= int(min_run):
            return int(f[i])
    return None


def stopping_metrics_for_speed(speed_kmh, reaction_time_sec=1.0, decel_mps2=6.5):
    """Compute stopping distance/time for a speed scenario."""
    v = float(speed_kmh) / 3.6
    reaction_distance = v * reaction_time_sec
    braking_distance = (v ** 2) / (2.0 * decel_mps2)
    stopping_distance = reaction_distance + braking_distance
    stopping_time = reaction_time_sec + (v / decel_mps2)
    return {
        "speed_kmh": float(speed_kmh),
        "speed_mps": v,
        "reaction_time_sec": float(reaction_time_sec),
        "decel_mps2": float(decel_mps2),
        "stopping_time_sec": float(stopping_time),
        "stopping_distance_m": float(stopping_distance),
    }


# Build predictions across test videos (uses causal temporal filtering in online mode by default)
pred_test = predict_from_df(
    df=df[df["video_id"].isin(test_video_ids)].copy(),
    clf=clf,
    cat_cols=cat_cols,
    num_cols=num_cols,
    threshold=DECISION_THRESHOLD,
    use_temporal_filter=ANALYSIS_USE_TEMPORAL,
    smoothing_mode=TEMP_SMOOTHING_MODE,
    smooth_window=TEMP_SMOOTH_WINDOW,
    enter_threshold=TEMP_ENTER_THRESHOLD,
    exit_threshold=TEMP_EXIT_THRESHOLD,
    min_run_length=TEMP_MIN_RUN_LENGTH,
 )

pred_col = "predicted_cross_temporal" if ANALYSIS_USE_TEMPORAL else "predicted_cross"

# Per-track onset analysis
track_rows = []
for (video_id, ped_id), g in pred_test.groupby(["video_id", "pedestrian_id"], sort=False):
    g = g.sort_values("frame_id")
    frames = g["frame_id"].to_numpy(dtype=int)
    gt = g["cross"].to_numpy(dtype=int)
    pr = g[pred_col].to_numpy(dtype=int)

    gt_onset = first_onset_frame(frames, gt, min_run=MIN_GT_ONSET_RUN)
    pr_onset = first_onset_frame(frames, pr, min_run=1)

    if gt_onset is None:
        # Ignore tracks that never actually cross in GT for lead-time evaluation
        continue

    lead_frames = None if pr_onset is None else int(gt_onset - pr_onset)
    lead_seconds = None if lead_frames is None else float(lead_frames / ANALYSIS_FPS)

    track_rows.append({
        "video_id": video_id,
        "pedestrian_id": ped_id,
        "gt_onset_frame": int(gt_onset),
        "pred_onset_frame": None if pr_onset is None else int(pr_onset),
        "lead_frames": lead_frames,
        "lead_seconds": lead_seconds,
        "predicted_before_or_on_time": None if lead_frames is None else int(lead_frames >= 0),
    })

lead_df = pd.DataFrame(track_rows)
if lead_df.empty:
    raise RuntimeError("No GT crossing-onset tracks found. Check labels or MIN_GT_ONSET_RUN.")

# Summary of lead time quality
n_tracks = len(lead_df)
n_detected = lead_df["pred_onset_frame"].notna().sum()
n_early = (lead_df["lead_frames"].fillna(-10**9) >= 0).sum()

summary = {
    "tracks_with_gt_crossing": int(n_tracks),
    "tracks_with_predicted_onset": int(n_detected),
    "coverage_rate": float(n_detected / n_tracks),
    "early_or_on_time_rate_all_tracks": float(n_early / n_tracks),
}
if n_detected > 0:
    detected = lead_df[lead_df["pred_onset_frame"].notna()].copy()
    summary.update({
        "lead_seconds_median_detected": float(detected["lead_seconds"].median()),
        "lead_seconds_p10_detected": float(detected["lead_seconds"].quantile(0.10)),
        "lead_seconds_p25_detected": float(detected["lead_seconds"].quantile(0.25)),
        "lead_seconds_p75_detected": float(detected["lead_seconds"].quantile(0.75)),
        "lead_seconds_p90_detected": float(detected["lead_seconds"].quantile(0.90)),
    })

print("Lead-time summary")
display(pd.DataFrame([summary]))

# Safety sufficiency table by speed scenario
scenario_rows = []
detected = lead_df[lead_df["pred_onset_frame"].notna()].copy()
for speed_kmh in SPEEDS_KMH:
    m = stopping_metrics_for_speed(
        speed_kmh=speed_kmh,
        reaction_time_sec=REACTION_TIME_SEC,
        decel_mps2=DECEL_MPS2,
    )
    required_lead_sec = m["stopping_time_sec"]
    if len(detected) > 0:
        sufficient = (detected["lead_seconds"] >= required_lead_sec).astype(int)
        sufficient_rate = float(sufficient.mean())
    else:
        sufficient_rate = 0.0

    scenario_rows.append({
        "speed_kmh": m["speed_kmh"],
        "required_lead_sec_for_stop": m["stopping_time_sec"],
        "required_stop_distance_m": m["stopping_distance_m"],
        "sufficient_stop_rate_over_detected_tracks": sufficient_rate,
    })

scenario_df = pd.DataFrame(scenario_rows)
print("Stopping sufficiency by speed scenario")
display(scenario_df)

# Optional: show tracks with least warning time
hard_cases = detected.sort_values("lead_seconds", ascending=True).head(20) if len(detected) > 0 else pd.DataFrame()
if not hard_cases.empty:
    print("Hardest cases (smallest lead time)")
    display(hard_cases[[
        "video_id", "pedestrian_id", "gt_onset_frame", "pred_onset_frame", "lead_frames", "lead_seconds"
    ]])

print("Assumptions used")
display(pd.DataFrame([{
    "fps": ANALYSIS_FPS,
    "prediction_column": pred_col,
    "smoothing_mode": TEMP_SMOOTHING_MODE if ANALYSIS_USE_TEMPORAL else "none",
    "reaction_time_sec": REACTION_TIME_SEC,
    "decel_mps2": DECEL_MPS2,
    "min_gt_onset_run": MIN_GT_ONSET_RUN,
}]))

Lead-time summary


C:\Users\ASUS\AppData\Local\Temp\ipykernel_7212\6579173.py:147: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,tracks_with_gt_crossing,tracks_with_predicted_onset,coverage_rate,early_or_on_time_rate_all_tracks,lead_seconds_median_detected,lead_seconds_p10_detected,lead_seconds_p25_detected,lead_seconds_p75_detected,lead_seconds_p90_detected
0,192,191,0.994792,0.838542,0.0,-0.3,0.0,1.266667,3.133333


Stopping sufficiency by speed scenario


,speed_kmh,required_lead_sec_for_stop,required_stop_distance_m,sufficient_stop_rate_over_detected_tracks
0,30.0,2.282051,13.675214,0.125654
1,40.0,2.709402,20.607787,0.109948
2,50.0,3.136752,28.727445,0.099476


...

,speed_kmh,required_lead_sec_for_stop,required_stop_distance_m,sufficient_stop_rate_over_detected_tracks
0,30.0,2.282051,13.675214,0.125654
1,40.0,2.709402,20.607787,0.109948
2,50.0,3.136752,28.727445,0.099476


Hardest cases (smallest lead time)


,video_id,pedestrian_id,gt_onset_frame,pred_onset_frame,lead_frames,lead_seconds
116,video_0224,0_224_1678b,41,122.0,-81.0,-2.700000
57,video_0141,0_141_872b,0,68.0,-68.0,-2.266667
117,video_0224,0_224_1680b,51,114.0,-63.0,-2.100000
58,video_0141,0_141_873b,9,57.0,-48.0,-1.600000
93,video_0187,0_187_1345b,0,41.0,-41.0,-1.366667
120,video_0234,0_234_1797b,32,65.0,-33.0,-1.100000
36,video_0113,0_113_642b,25,56.0,-31.0,-1.033333
94,video_0187,0_187_1346b,0,30.0,-30.0,-1.000000
179,video_0327,0_327_2583b,34,63.0,-29.0,-0.966667
154,video_0305,0_305_2362b,0,28.0,-28.0,-0.933333


Assumptions used


,fps,prediction_column,smoothing_mode,reaction_time_sec,decel_mps2,min_gt_onset_run
0,30.0,predicted_cross_temporal,online,1.0,6.5,3


In [27]:
# Hyperparameter sweep: maximize early lead time while controlling false alarms (validation split)
import numpy as np
import pandas as pd

# -------------------- Sweep Config --------------------
SWEEP_THRESHOLDS = [0.15, 0.20, 0.25, 0.30, 0.35]
SWEEP_WINDOWS = [5, 7, 9]
SWEEP_ENTER_EXIT_GAPS = [0.05, 0.10]  # exit = enter - gap
SWEEP_MIN_RUNS = [2, 3, 4]
SWEEP_SMOOTHING_MODE = "online"  # keep causal for real-time

# Optional: stopping scenario used in objective penalty
SWEEP_SPEED_KMH_FOR_OBJECTIVE = 30.0
SWEEP_REACTION_TIME_SEC = REACTION_TIME_SEC
SWEEP_DECEL_MPS2 = DECEL_MPS2

# Objective weights (tune as needed)
W_EARLY_RATE = 2.0
W_MEDIAN_LEAD = 0.8
W_P25_LEAD = 0.6
W_FALSE_ALARM = 1.2
W_MISS_RATE = 1.0
W_STOP_SUFFICIENCY = 0.6


def _first_onset_frame_local(frame_ids, labels, min_run=1):
    """Local onset helper to keep this sweep cell self-contained."""
    f = np.asarray(frame_ids, dtype=int)
    y = np.asarray(labels, dtype=int)
    n = len(y)
    if n == 0:
        return None
    for i in range(n):
        if y[i] != 1:
            continue
        if i > 0 and y[i - 1] == 1:
            continue
        j = i
        while j < n and y[j] == 1:
            j += 1
        if (j - i) >= int(min_run):
            return int(f[i])
    return None


def _stop_time_local(speed_kmh, reaction_time_sec=1.0, decel_mps2=6.5):
    v = float(speed_kmh) / 3.6
    return float(reaction_time_sec + (v / decel_mps2))


def evaluate_setting_on_val(
    threshold, smooth_window, enter_th, exit_th, min_run_length, smoothing_mode="online"
 ):
    val_subset = df[df["video_id"].isin(val_videos)].copy()
    pred_val = predict_from_df(
        df=val_subset,
        clf=clf,
        cat_cols=cat_cols,
        num_cols=num_cols,
        threshold=threshold,
        use_temporal_filter=True,
        smoothing_mode=smoothing_mode,
        smooth_window=smooth_window,
        enter_threshold=enter_th,
        exit_threshold=exit_th,
        min_run_length=min_run_length,
    )

    lead_seconds = []
    n_gt_tracks = 0
    n_detected_gt = 0
    n_early_gt = 0
    n_non_cross_tracks = 0
    n_false_alarm_tracks = 0

    for (_, _), g in pred_val.groupby(["video_id", "pedestrian_id"], sort=False):
        g = g.sort_values("frame_id")
        frames = g["frame_id"].to_numpy(dtype=int)
        gt = g["cross"].to_numpy(dtype=int)
        pr = g["predicted_cross_temporal"].to_numpy(dtype=int)

        gt_onset = _first_onset_frame_local(frames, gt, min_run=MIN_GT_ONSET_RUN)
        pr_onset = _first_onset_frame_local(frames, pr, min_run=1)

        if gt_onset is None:
            n_non_cross_tracks += 1
            if pr_onset is not None:
                n_false_alarm_tracks += 1
            continue

        n_gt_tracks += 1
        if pr_onset is not None:
            n_detected_gt += 1
            lead = (gt_onset - pr_onset) / ANALYSIS_FPS
            lead_seconds.append(float(lead))
            if lead >= 0:
                n_early_gt += 1

    detection_rate = float(n_detected_gt / n_gt_tracks) if n_gt_tracks > 0 else 0.0
    early_rate_all_gt = float(n_early_gt / n_gt_tracks) if n_gt_tracks > 0 else 0.0
    early_rate_detected = float(n_early_gt / n_detected_gt) if n_detected_gt > 0 else 0.0
    miss_rate_gt = float(1.0 - detection_rate) if n_gt_tracks > 0 else 1.0
    false_alarm_rate_non_cross = (
        float(n_false_alarm_tracks / n_non_cross_tracks) if n_non_cross_tracks > 0 else 0.0
    )

    if len(lead_seconds) > 0:
        lead_arr = np.asarray(lead_seconds, dtype=float)
        lead_median = float(np.median(lead_arr))
        lead_p25 = float(np.quantile(lead_arr, 0.25))
        lead_p10 = float(np.quantile(lead_arr, 0.10))
    else:
        lead_median = np.nan
        lead_p25 = np.nan
        lead_p10 = np.nan

    required_stop_lead = _stop_time_local(
        speed_kmh=SWEEP_SPEED_KMH_FOR_OBJECTIVE,
        reaction_time_sec=SWEEP_REACTION_TIME_SEC,
        decel_mps2=SWEEP_DECEL_MPS2,
    )
    if len(lead_seconds) > 0:
        stop_sufficient_rate = float((np.asarray(lead_seconds) >= required_stop_lead).mean())
    else:
        stop_sufficient_rate = 0.0

    lead_median_pos = max(0.0, 0.0 if np.isnan(lead_median) else lead_median)
    lead_p25_pos = max(0.0, 0.0 if np.isnan(lead_p25) else lead_p25)

    objective = (
        W_EARLY_RATE * early_rate_all_gt
        + W_MEDIAN_LEAD * lead_median_pos
        + W_P25_LEAD * lead_p25_pos
        + W_STOP_SUFFICIENCY * stop_sufficient_rate
        - W_FALSE_ALARM * false_alarm_rate_non_cross
        - W_MISS_RATE * miss_rate_gt
    )

    return {
        "objective": float(objective),
        "threshold": float(threshold),
        "smooth_window": int(smooth_window),
        "enter_threshold": float(enter_th),
        "exit_threshold": float(exit_th),
        "min_run_length": int(min_run_length),
        "smoothing_mode": smoothing_mode,
        "n_gt_tracks": int(n_gt_tracks),
        "n_non_cross_tracks": int(n_non_cross_tracks),
        "detection_rate": detection_rate,
        "early_rate_all_gt": early_rate_all_gt,
        "early_rate_detected": early_rate_detected,
        "miss_rate_gt": miss_rate_gt,
        "false_alarm_rate_non_cross": false_alarm_rate_non_cross,
        "lead_median_sec": lead_median,
        "lead_p25_sec": lead_p25,
        "lead_p10_sec": lead_p10,
        "stop_sufficient_rate": stop_sufficient_rate,
    }


rows = []
for th in SWEEP_THRESHOLDS:
    for win in SWEEP_WINDOWS:
        for gap in SWEEP_ENTER_EXIT_GAPS:
            enter_th = float(th)
            exit_th = max(0.0, float(th) - float(gap))
            for min_run in SWEEP_MIN_RUNS:
                rows.append(
                    evaluate_setting_on_val(
                        threshold=th,
                        smooth_window=win,
                        enter_th=enter_th,
                        exit_th=exit_th,
                        min_run_length=min_run,
                        smoothing_mode=SWEEP_SMOOTHING_MODE,
                    )
                )

sweep_df = pd.DataFrame(rows).sort_values("objective", ascending=False).reset_index(drop=True)
print(f"Evaluated {len(sweep_df)} parameter combinations on validation split.")

cols_show = [
    "objective",
    "threshold", "smooth_window", "enter_threshold", "exit_threshold", "min_run_length",
    "detection_rate", "early_rate_all_gt", "early_rate_detected",
    "lead_p10_sec", "lead_p25_sec", "lead_median_sec",
    "false_alarm_rate_non_cross", "stop_sufficient_rate",
]
display(sweep_df[cols_show].head(15))

if not sweep_df.empty:
    best = sweep_df.iloc[0]
    print("\nBest config suggestion (validation):")
    print({
        "DECISION_THRESHOLD": float(best["threshold"]),
        "TEMP_SMOOTH_WINDOW": int(best["smooth_window"]),
        "TEMP_ENTER_THRESHOLD": float(best["enter_threshold"]),
        "TEMP_EXIT_THRESHOLD": float(best["exit_threshold"]),
        "TEMP_MIN_RUN_LENGTH": int(best["min_run_length"]),
        "TEMP_SMOOTHING_MODE": str(best["smoothing_mode"]),
    })
    print("\nCopy these values into Cell 6 config, then rerun Cell 6 and Cell 7/8 for confirmation.")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_7212\6579173.py:147: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7212\6579173.py:147: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7212\6579173.py:147: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future ver

C:\Users\ASUS\AppData\Local\Temp\ipykernel_7212\6579173.py:147: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7212\6579173.py:147: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7212\6579173.py:147: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future ver

Evaluated 90 parameter combinations on validation split.


,objective,threshold,smooth_window,enter_threshold,exit_threshold,min_run_length,detection_rate,early_rate_all_gt,early_rate_detected,lead_p10_sec,lead_p25_sec,lead_median_sec,false_alarm_rate_non_cross,stop_sufficient_rate
0,1.574447,0.15,5,0.15,0.10,2,1.0,0.972973,0.972973,0.000000,0.0,0.0,0.363636,0.108108
1,1.574447,0.15,5,0.15,0.10,3,1.0,0.972973,0.972973,0.000000,0.0,0.0,0.363636,0.108108
2,1.574447,0.15,5,0.15,0.10,4,1.0,0.972973,0.972973,0.000000,0.0,0.0,0.363636,0.108108
3,1.574447,0.15,5,0.15,0.05,2,1.0,0.972973,0.972973,0.000000,0.0,0.0,0.363636,0.108108
4,1.574447,0.15,5,0.15,0.05,3,1.0,0.972973,0.972973,0.000000,0.0,0.0,0.363636,0.108108
5,1.574447,0.15,5,0.15,0.05,4,1.0,0.972973,0.972973,0.000000,0.0,0.0,0.363636,0.108108
6,1.142015,0.15,7,0.15,0.10,2,1.0,0.756757,0.756757,-0.033333,0.0,0.0,0.363636,0.108108
7,1.142015,0.15,7,0.15,0.10,3,1.0,0.756757,0.756757,-0.033333,0.0,0.0,0.363636,0.108108
8,1.142015,0.15,7,0.15,0.10,4,1.0,0.756757,0.756757,-0.033333,0.0,0.0,0.363636,0.108108
9,1.142015,0.15,7,0.15,0.05,2,1.0,0.756757,0.756757,-0.033333,0.0,0.0,0.363636,0.108108



Best config suggestion (validation):
{'DECISION_THRESHOLD': 0.15, 'TEMP_SMOOTH_WINDOW': 5, 'TEMP_ENTER_THRESHOLD': 0.15, 'TEMP_EXIT_THRESHOLD': 0.09999999999999999, 'TEMP_MIN_RUN_LENGTH': 2, 'TEMP_SMOOTHING_MODE': 'online'}

Copy these values into Cell 6 config, then rerun Cell 6 and Cell 7/8 for confirmation.
